# Feature Engineering for ML

This notebook constructs event-level features for the supervised learning problem introduced by event labeling. Each row corresponds to a candidate spread event. The feature values must be observable at the event date. The target label may use the future path because it is the outcome being learned, but the inputs must not use future information.

## Research framing

The label is not a next-day stock-return label. It is the event outcome:

$$P(\text{residual mean-reverts before stop-loss}\mid\text{current setup features}).$$

The feature matrix therefore describes the spread state, relationship stability, volatility regime, recent market movement, and optional volume pressure at signal time.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.features import FeatureConfig, create_feature_outputs
from src.database import (
    connect_database,
    initialize_database,
    store_event_feature_matrix,
    store_feature_summary_statistics,
    store_feature_missingness_report,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURE_DIR = PROJECT_ROOT / 'figures'
SQL_SCHEMA = PROJECT_ROOT / 'sql' / 'schema.sql'
DATABASE_PATH = PROJECT_ROOT / 'data' / 'triangular_stat_arb.sqlite'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

## Load event and residual inputs

The notebook expects event labels and dynamic residual outputs from the labeling and hedge-ratio notebooks. If local files are not present, the packaged placeholder CSVs show the expected schema only. Real project runs should overwrite them from the actual database or prior notebooks.

In [ ]:
candidate_events_path = DATA_DIR / 'candidate_events_table.csv'
event_labels_path = DATA_DIR / 'event_labels_table.csv'
residual_path_candidates = [
    DATA_DIR / 'kalman_residual_table.csv',
    DATA_DIR / 'dynamic_residual_table.csv',
]

candidate_events = pd.read_csv(candidate_events_path, parse_dates=['event_date'])
event_labels = pd.read_csv(event_labels_path, parse_dates=['event_date'], infer_datetime_format=False)

residuals = None
for path in residual_path_candidates:
    if path.exists():
        residuals = pd.read_csv(path, parse_dates=['date'])
        break
if residuals is None:
    raise FileNotFoundError('No residual table found. Run the rolling/ridge or Kalman hedge-ratio notebook first.')

## Optional market inputs

The feature builder accepts returns, log prices, and volumes when available. Missing optional inputs produce missing feature values instead of fabricated values.

In [ ]:
def read_optional_csv(path, date_col='date'):
    if path.exists():
        return pd.read_csv(path, parse_dates=[date_col])
    return None

returns = read_optional_csv(DATA_DIR / 'returns.csv')
log_prices = read_optional_csv(DATA_DIR / 'log_prices.csv')
volumes = read_optional_csv(DATA_DIR / 'volumes.csv')
coefficients = read_optional_csv(DATA_DIR / 'kalman_hedge_ratio_table.csv')

In [ ]:
config = FeatureConfig(
    residual_window=30,
    volatility_window=20,
    stability_window=20,
    correlation_window=30,
    moving_average_window=30,
    min_periods=10,
    market_symbol='QQQ',
)

outputs = create_feature_outputs(
    candidate_events=candidate_events,
    residuals=residuals,
    labels=event_labels,
    returns=returns,
    log_prices=log_prices,
    volumes=volumes,
    coefficients=coefficients,
    config=config,
)

feature_matrix = outputs['feature_matrix']
summary_statistics = outputs['feature_summary_statistics']
missingness_report = outputs['feature_missingness_report']
correlation_matrix = outputs['feature_correlation_matrix']

feature_matrix.head()

## Feature diagnostics

The missingness report is a leakage and data-quality check. Optional volume features should be missing when volume data is unavailable. Core residual features should not be missing after the initial rolling windows.

In [ ]:
feature_matrix.to_csv(DATA_DIR / 'event_feature_matrix.csv', index=False)
summary_statistics.to_csv(DATA_DIR / 'feature_summary_statistics.csv', index=False)
missingness_report.to_csv(DATA_DIR / 'feature_missingness_report.csv', index=False)
correlation_matrix.to_csv(DATA_DIR / 'feature_correlation_matrix.csv')

missingness_report.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
if not correlation_matrix.empty:
    matrix = correlation_matrix.fillna(0.0).to_numpy()
    im = ax.imshow(matrix, aspect='auto')
    ax.set_xticks(range(len(correlation_matrix.columns)))
    ax.set_yticks(range(len(correlation_matrix.index)))
    ax.set_xticklabels(correlation_matrix.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(correlation_matrix.index, fontsize=7)
    ax.set_title('Feature Correlation Heatmap')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
else:
    ax.text(0.5, 0.5, 'Insufficient numeric feature variation', ha='center', va='center')
    ax.set_axis_off()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'feature_correlation_heatmap.png', dpi=160)
plt.show()

## SQL storage

The feature matrix is stored separately from labels and residuals. This preserves auditability: event construction, outcome labeling, and feature engineering remain separate reproducible steps.

In [ ]:
initialize_database(DATABASE_PATH, SQL_SCHEMA)
with connect_database(DATABASE_PATH) as conn:
    store_event_feature_matrix(conn, feature_matrix, if_exists='replace')
    store_feature_summary_statistics(conn, summary_statistics, if_exists='replace')
    store_feature_missingness_report(conn, missingness_report, if_exists='replace')

feature_matrix.shape